# Build Analysis Frame

Constructs the central DataFrame for the analysis. Primary key: `(match_id, player_id)`.

## 0. Imports & data

In [1]:
import warnings
warnings.filterwarnings("ignore")

import re
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path("data")

events  = pd.read_parquet(DATA_DIR / "events.parquet")
matches = pd.read_parquet(DATA_DIR / "matches.parquet")

events["player_id"] = pd.to_numeric(events["player_id"], errors="coerce")

# ---------------------------------------------------------------------------
# Tackle-only duels — deliberate defensive action only
# ---------------------------------------------------------------------------
# StatsBomb's 'Duel' event mixes Tackles (active dispossession attempts) with
# 'Aerial Lost' (the losing side of an aerial; the winning side is recorded on
# the resuming event with aerial_won=True, not as a Duel). To focus on actions
# the focal player actively chooses, we re-label tackles as their own event
# type and drop aerial-lost duels from all per-type counts.
_is_tackle = (events["type"] == "Duel") & (events["duel_type"] == "Tackle")
events.loc[_is_tackle, "type"] = "Tackle"
_is_aerial = (events["type"] == "Duel") & (events["duel_type"] == "Aerial Lost")
events.loc[_is_aerial, "type"] = "Aerial Lost"
print(f"Re-labelled {_is_tackle.sum():,} tackles and {_is_aerial.sum():,} aerial-lost duels; the bare 'Duel' label no longer appears in the events table.")

# ---------------------------------------------------------------------------
# Analysis windows  —  single source of truth
# ---------------------------------------------------------------------------
# pre   : 1st-half pre-treatment, period 1, minutes [0, 15)   covariate window
# treat : 1st-half treatment,     period 1, minutes [15, 45]  yellow-card window
# post  : 2nd-half outcome,       period 2, minutes [45, 60]  defensive-behaviour window
WINDOWS = {
    "pre"  : dict(period=1, lo=0,  hi=15, hi_inclusive=False),
    "treat": dict(period=1, lo=15, hi=45, hi_inclusive=True),
    "post" : dict(period=2, lo=45, hi=60, hi_inclusive=True),
}


def in_window(events: pd.DataFrame, name: str) -> pd.Series:
    """Boolean mask: events falling inside `WINDOWS[name]`."""
    w = WINDOWS[name]
    minute = events["minute"]
    upper = (minute <= w["hi"]) if w["hi_inclusive"] else (minute < w["hi"])
    return (events["period"] == w["period"]) & (minute >= w["lo"]) & upper


def clean_event_name(name: str) -> str:
    """Normalise a StatsBomb event-type / column label to a snake_case identifier.

    Lower-cases and collapses any run of non-alphanumeric characters to a single
    underscore (so 'Ball Receipt*' → 'ball_receipt', 'Referee Ball-Drop' →
    'referee_ball_drop', '50/50' → '50_50').
    """
    return re.sub(r"[^0-9a-z]+", "_", name.lower()).strip("_")


def anti_join(df: pd.DataFrame, exclude: pd.DataFrame,
              on=("match_id", "player_id")) -> pd.DataFrame:
    """Rows of `df` whose `on`-keys are NOT present in `exclude`."""
    on = list(on)
    merged = df.merge(exclude[on].drop_duplicates(), on=on,
                      how="left", indicator=True)
    return merged.loc[merged["_merge"] == "left_only"].drop(columns="_merge")


print(f"Events : {len(events):>12,}")
print(f"Matches: {len(matches):>12,}")

Re-labelled 134,238 tackles and 123,623 aerial-lost duels; the bare 'Duel' label no longer appears in the events table.
Events :   12,188,949
Matches:        3,464


## 1. Base frame — `(match_id, player_id)`

All player × match combinations observed in the event stream.

In [2]:
player_events = events.dropna(subset=["player_id"])

base = (
    player_events
    .groupby(["match_id", "player_id"], sort=False)
    .agg(team_id=("team_id", "first"),
         player_name=("player", "first"))
    .reset_index()
)

print(f"Base frame: {len(base):,} player × match rows")
base.head(3)

Base frame: 97,581 player × match rows


,match_id,player_id,team_id,player_name
0,3895302,34870.0,176,Nick Woltemade
1,3895302,12299.0,176,Marvin Ducksch
2,3895302,31100.0,176,Christian Groß


## 2. Eligibility criteria

Each criterion is a function returning the `(match_id, player_id)` pairs to **reject**. Applied sequentially via anti-join.

In [3]:
# ---------------------------------------------------------------------------
# Card masks
# ---------------------------------------------------------------------------

def _yellow_card_mask(events: pd.DataFrame) -> pd.Series:
    """Yellow card from either a foul or a stand-alone Bad Behaviour event."""
    return (
        (events["foul_committed_card"] == "Yellow Card")
        | (events["bad_behaviour_card"] == "Yellow Card")
    )


def _red_or_second_yellow_mask(events: pd.DataFrame) -> pd.Series:
    return (
        events["foul_committed_card"].isin(["Red Card", "Second Yellow"])
        | events["bad_behaviour_card"].isin(["Red Card", "Second Yellow"])
    )


def _player_pairs(events: pd.DataFrame, mask: pd.Series) -> pd.DataFrame:
    """Distinct (match_id, player_id) pairs where `mask` is True."""
    return (
        events.loc[mask, ["match_id", "player_id"]]
              .dropna(subset=["player_id"])
              .drop_duplicates()
    )


# ---------------------------------------------------------------------------
# Each criterion returns the (match_id, player_id) pairs to REJECT.
# ---------------------------------------------------------------------------

def reject_substituted_on(events: pd.DataFrame) -> pd.DataFrame:
    """(a) Was substituted in — i.e. did NOT start the match."""
    return _player_pairs(events, events["type"] == "Player On")


def reject_early_exit(events: pd.DataFrame) -> pd.DataFrame:
    """(a) Left the pitch before minute 60 (sub off, player off, red, 2nd yellow)."""
    mask = (events["minute"] < 60) & (
        events["type"].isin(["Substitution", "Player Off"])
        | _red_or_second_yellow_mask(events)
    )
    return _player_pairs(events, mask)


def reject_yellow_in_pre(events: pd.DataFrame) -> pd.DataFrame:
    """(b) Got a yellow card in the pre-treatment window."""
    return _player_pairs(events, in_window(events, "pre") & _yellow_card_mask(events))


def reject_multi_yellow_in_treat(events: pd.DataFrame) -> pd.DataFrame:
    """(c) Got more than one yellow card in the treatment window."""
    yc = events.loc[in_window(events, "treat") & _yellow_card_mask(events)]
    return (
        yc.dropna(subset=["player_id"])
          .groupby(["match_id", "player_id"]).size()
          .reset_index(name="_n")
          .query("_n >= 2")[["match_id", "player_id"]]
    )


def reject_yellow_in_post(events: pd.DataFrame) -> pd.DataFrame:
    """(d) Got a yellow card in the post-treatment window."""
    return _player_pairs(events, in_window(events, "post") & _yellow_card_mask(events))


# ---------------------------------------------------------------------------
# Registry — comment out a line to disable a criterion
# ---------------------------------------------------------------------------

ELIGIBILITY_CRITERIA = {
    "is_starter"          : reject_substituted_on,
    "plays_60_plus_min"   : reject_early_exit,
    "no_yellow_in_pre"    : reject_yellow_in_pre,
    "<=1_yellow_in_treat" : reject_multi_yellow_in_treat,
    "no_yellow_in_post"   : reject_yellow_in_post,
}


def apply_eligibility(base: pd.DataFrame, events: pd.DataFrame,
                      criteria: dict) -> pd.DataFrame:
    """Sequentially anti-join each rejection set; print an attrition table."""
    df = base.copy()
    print(f"{'Criterion':<28} {'Remaining':>10} {'Dropped':>10}")
    print("-" * 50)
    print(f"{'(base frame)':<28} {len(df):>10,}")
    for name, reject_fn in criteria.items():
        prev = len(df)
        df = anti_join(df, reject_fn(events))
        print(f"{name:<28} {len(df):>10,} {prev - len(df):>10,}")
    print("-" * 50)
    print(f"{'Final eligible sample':<28} {len(df):>10,}")
    return df


base_eligible = apply_eligibility(base, events, ELIGIBILITY_CRITERIA)

Criterion                     Remaining    Dropped
--------------------------------------------------
(base frame)                     97,581
is_starter                       94,415      3,166


plays_60_plus_min                89,884      4,531
no_yellow_in_pre                 89,261        623


<=1_yellow_in_treat              89,261          0
no_yellow_in_post                87,157      2,104
--------------------------------------------------
Final eligible sample            87,157


## 3. Treatment

`treat_yellow_card` = 1 if the player received a yellow card in the treatment window (period 1, minutes 15-45).

In [4]:
def build_treat_yellow_card(events: pd.DataFrame) -> pd.DataFrame:
    """(match_id, player_id, treat_yellow_card=1) for players carded in the
    treatment window."""
    pairs = _player_pairs(events, in_window(events, "treat") & _yellow_card_mask(events))
    return pairs.assign(treat_yellow_card=1)


treat_yc = build_treat_yellow_card(events)
base_eligible = base_eligible.merge(treat_yc, on=["match_id", "player_id"], how="left")
base_eligible["treat_yellow_card"] = base_eligible["treat_yellow_card"].fillna(0).astype(int)

print(f"Treated players: {base_eligible['treat_yellow_card'].sum():,}")
print(f"Treatment rate : {base_eligible['treat_yellow_card'].mean():.2%}")

Treated players: 3,265
Treatment rate : 3.75%


## 4. Outcomes — post-window event counts

The post-window counts mirror the pre-side schema (consistent definitions throughout):

* **Overall** (`post_n_events`): every player-attributed event in the post window.
* **Defensive engagement** (`post_n_def_events`): sum of the three deliberate
  defensive actions — `Pressure`, `Tackle`, `Foul Committed`. Used as a fourth,
  aggregate DV alongside the three components.
* **Offensive** (`post_n_off_events`): sum over `Pass`, `Carry`, `Ball Receipt*`,
  `Dribble`, `Shot`, `Foul Won`.

Per-type columns are retained for the full defensive set (incl. `Aerial Lost`,
`Interception`, `Block`, `50/50`, `Clearance`) and offensive failures
(`Dispossessed`, `Dribbled Past`, `Error`); the latter are *not* included in any
engagement aggregate because they happen *to* the player.

Note. StatsBomb's `Duel` event mixes Tackles with `Aerial Lost`. We re-label
both in cell 2 so the events table no longer contains a bare `Duel` type, and
all per-type counts give clean `*_n_tackle` and `*_n_aerial_lost` columns.

In [5]:
# All defensive event types: kept as per-type columns. Aerial Lost is the
# losing side of an aerial contest (the winning side is logged on the
# resuming event with aerial_won=True). It is NOT an action the player
# actively chooses, so it is excluded from the headline DV aggregate.
DEFENSIVE_EVENT_TYPES = [
    "Pressure",
    "Tackle",
    "Foul Committed",
    "Aerial Lost",
    "Interception",
    "Block",
    "50/50",
    "Clearance",
]

# Headline DV set: the three deliberate defensive actions.
# `post_n_def_events` is their sum, used as a fourth (engagement) DV.
DV_TYPES = ["Pressure", "Tackle", "Foul Committed"]

# Outcomes that happen TO the player (not actions they initiate). Kept as
# per-type columns but excluded from the `n_def_events` engagement aggregate.
DEFENSIVE_FAILURE_EVENT_TYPES = [
    "Dispossessed",
    "Dribbled Past",
    "Error",
]

OFFENSIVE_EVENT_TYPES = [
    "Pass",
    "Carry",
    "Ball Receipt*",
    "Dribble",
    "Shot",
    "Foul Won",
]

# All per-type buckets materialised on both pre and post sides.
COUNTED_EVENT_TYPES = (
    DEFENSIVE_EVENT_TYPES
    + DEFENSIVE_FAILURE_EVENT_TYPES
    + OFFENSIVE_EVENT_TYPES
)


def build_post_counts(events: pd.DataFrame) -> pd.DataFrame:
    """(match_id, player_id) post-window counts.

    Materialises one `post_n_<type>` column for each type in
    COUNTED_EVENT_TYPES plus three aggregates:
      * `post_n_events`     — all player events in the post window
      * `post_n_def_events` — sum over DEFENSIVE_EVENT_TYPES (active engagement)
      * `post_n_off_events` — sum over OFFENSIVE_EVENT_TYPES

    Defensive failures keep per-type columns but are NOT included in the
    `n_def_events` aggregate.
    """
    pp = events.dropna(subset=["player_id"])
    post = pp.loc[in_window(pp, "post")]

    sub = post.loc[post["type"].isin(COUNTED_EVENT_TYPES)]
    counts = (
        sub.groupby(["match_id", "player_id", "type"], sort=False).size()
           .unstack(fill_value=0)
    )
    for t in COUNTED_EVENT_TYPES:
        if t not in counts.columns:
            counts[t] = 0
    counts = counts[COUNTED_EVENT_TYPES]
    counts.columns = [f"post_n_{clean_event_name(c)}" for c in counts.columns]

    def_cols = [f"post_n_{clean_event_name(c)}" for c in DEFENSIVE_EVENT_TYPES]
    off_cols = [f"post_n_{clean_event_name(c)}" for c in OFFENSIVE_EVENT_TYPES]
    dv_cols = [f"post_n_{clean_event_name(t)}" for t in DV_TYPES if f"post_n_{clean_event_name(t)}" in counts.columns]
    counts["post_n_def_events"] = counts[dv_cols].sum(axis=1)
    counts["post_n_off_events"] = counts[off_cols].sum(axis=1)
    counts = counts.reset_index()

    totals = (
        post.groupby(["match_id", "player_id"]).size()
            .rename("post_n_events").reset_index()
    )
    return counts.merge(totals, on=["match_id", "player_id"], how="outer")


post_counts = build_post_counts(events)
post_cols   = [c for c in post_counts.columns if c.startswith("post_")]

base_eligible = base_eligible.merge(post_counts, on=["match_id", "player_id"], how="left")
base_eligible[post_cols] = base_eligible[post_cols].fillna(0).astype(int)

agg_cols      = ["post_n_events", "post_n_def_events", "post_n_off_events"]
per_type_cols = sorted(set(post_cols) - set(agg_cols))

print(f"{'Metric':<26} {'Treated':>10} {'Control':>10} {'Diff':>8}")
print("-" * 56)
for col in agg_cols + per_type_cols:
    t = base_eligible.loc[base_eligible["treat_yellow_card"] == 1, col].mean()
    c = base_eligible.loc[base_eligible["treat_yellow_card"] == 0, col].mean()
    print(f"{col:<26} {t:>10.3f} {c:>10.3f} {t - c:>+8.3f}")


Metric                        Treated    Control     Diff
--------------------------------------------------------
post_n_events                  27.823     21.364   +6.459
post_n_def_events               3.319      2.359   +0.960
post_n_off_events              21.696     16.765   +4.931
post_n_50_50                    0.027      0.019   +0.008
post_n_aerial_lost              0.276      0.204   +0.072
post_n_ball_receipt             7.062      5.576   +1.486
post_n_block                    0.331      0.233   +0.098
post_n_carry                    6.042      4.652   +1.391
post_n_clearance                0.399      0.268   +0.131
post_n_dispossessed             0.199      0.157   +0.042
post_n_dribble                  0.271      0.221   +0.049
post_n_dribbled_past            0.200      0.136   +0.063
post_n_error                    0.004      0.003   +0.001
post_n_foul_committed           0.199      0.151   +0.048
post_n_foul_won                 0.240      0.163   +0.077
post_n_intercep

## 5. Match-level covariates

In [6]:
COMPETITION_TYPE = {
    # Domestic leagues
    "1. Bundesliga"          : "Domestic League",
    "La Liga"                : "Domestic League",
    "Premier League"         : "Domestic League",
    "Ligue 1"                : "Domestic League",
    "Serie A"                : "Domestic League",
    "FA Women's Super League": "Domestic League",
    "Indian Super league"    : "Domestic League",
    "Liga Profesional"       : "Domestic League",
    "Major League Soccer"    : "Domestic League",
    "North American League"  : "Domestic League",
    "NWSL"                   : "Domestic League",
    # Domestic cups
    "Copa del Rey"           : "Domestic Cup",
    # International club
    "Champions League"       : "International Club",
    "UEFA Europa League"     : "International Club",
    # International national team
    "FIFA World Cup"         : "International NT",
    "FIFA U20 World Cup"     : "International NT",
    "UEFA Euro"              : "International NT",
    "UEFA Women's Euro"      : "International NT",
    "Copa America"           : "International NT",
    "African Cup of Nations" : "International NT",
    "Women's World Cup"      : "International NT",
}

def build_match_covariates(matches: pd.DataFrame) -> pd.DataFrame:
    """match_id-keyed: competition_type, competition, season, gender,
    match_date, match_week."""
    df = matches[[
        "match_id", "competition_name", "season_name",
        "competition_gender", "match_date", "match_week",
    ]].copy().rename(columns={
        "competition_name"  : "competition",
        "season_name"       : "season",
        "competition_gender": "gender",
    })
    df["competition_type"] = df["competition"].map(COMPETITION_TYPE)
    df["match_date"] = pd.to_datetime(df["match_date"]).dt.date
    df["match_week"] = df["match_week"].replace(0, pd.NA).astype("Int64")
    return df


match_covs = build_match_covariates(matches)
base_eligible = base_eligible.merge(match_covs, on="match_id", how="left")

print("Competition type distribution:")
print(base_eligible["competition_type"].value_counts().to_string())
print("\nGender distribution:")
print(base_eligible["gender"].value_counts().to_string())
print(f"\nmatch_date range: {base_eligible['match_date'].min()} – {base_eligible['match_date'].max()}")
print(f"match_week range: {base_eligible['match_week'].min()} – {base_eligible['match_week'].max()}")

Competition type distribution:
competition_type
Domestic League       72974
International NT      13615
International Club      505
Domestic Cup             63

Gender distribution:
gender
male      73377
female    13780

match_date range: 1958-06-24 – 2025-07-27
match_week range: 1 – 38


### 5.1 Pre-window event intensity (team / opponent)

Counts of each event type in the pre window for the player's own team (`pre_team_*`) and the opponent (`pre_opp_*`). Proxy for early-game tempo and territorial pressure.

In [7]:
# Structural / broadcast markers — not real on-pitch events. Excluded from
# both per-type counts and the n_events total.
# Note: StatsBomb's casing is inconsistent — 'Camera On' vs 'Camera off'.
PRE_EXCLUDE_EVENT_TYPES = {"Camera On", "Camera off", "Half Start", "Starting XI"}


def build_pre_team_intensity(events: pd.DataFrame) -> pd.DataFrame:
    """(match_id, team_id) event-type counts in the pre window.

    Returns one row per (match_id, team_id) with bare metric column names
    (`n_events`, `n_pass`, `n_carry`, …) — the caller prefixes them with
    `pre_team_` or `pre_opp_` after merging.
    """
    early = events.loc[
        in_window(events, "pre")
        & ~events["type"].isin(PRE_EXCLUDE_EVENT_TYPES)
    ]
    by_type = (
        early.groupby(["match_id", "team_id", "type"]).size()
             .unstack(fill_value=0)
    )
    by_type.columns = [f"n_{clean_event_name(c)}" for c in by_type.columns]
    return by_type.reset_index()


def _opponent_team_map(matches: pd.DataFrame) -> pd.DataFrame:
    """(match_id, team_id) → opponent_team_id."""
    home = matches[["match_id", "home_team_id", "away_team_id"]].rename(
        columns={"home_team_id": "team_id", "away_team_id": "opponent_team_id"}
    )
    away = matches[["match_id", "away_team_id", "home_team_id"]].rename(
        columns={"away_team_id": "team_id", "home_team_id": "opponent_team_id"}
    )
    return pd.concat([home, away], ignore_index=True)


def build_window_goals(events: pd.DataFrame, window_name: str) -> pd.DataFrame:
    """(match_id, team_id) -> goals scored in the named analysis window.

    Counts shot goals (Shot with shot_outcome=='Goal') and 'Own Goal For'
    events (StatsBomb codes the benefiting team in `team_id`).
    """
    in_w = in_window(events, window_name)
    shot_goals = events.loc[
        in_w & (events["type"] == "Shot") & (events["shot_outcome"] == "Goal"),
        ["match_id", "team_id"],
    ]
    og_for = events.loc[
        in_w & (events["type"] == "Own Goal For"), ["match_id", "team_id"]
    ]
    return (
        pd.concat([shot_goals, og_for], ignore_index=True)
          .groupby(["match_id", "team_id"]).size()
          .reset_index(name="goals")
    )


pre_intensity = build_pre_team_intensity(events)
metric_cols = [c for c in pre_intensity.columns if c.startswith("n_")]

# --- own team ----------------------------------------------------------
own = pre_intensity.rename(columns={c: f"pre_team_{c}" for c in metric_cols})
base_eligible = base_eligible.merge(own, on=["match_id", "team_id"], how="left")

# --- opponent ----------------------------------------------------------
opp_map = _opponent_team_map(matches)
opp = pre_intensity.rename(
    columns={"team_id": "opponent_team_id",
             **{c: f"pre_opp_{c}" for c in metric_cols}}
)
base_eligible = (
    base_eligible
    .merge(opp_map, on=["match_id", "team_id"], how="left")
    .merge(opp, on=["match_id", "opponent_team_id"], how="left")
)

pre_team_cols = [f"pre_team_{c}" for c in metric_cols]
pre_opp_cols  = [f"pre_opp_{c}"  for c in metric_cols]
base_eligible[pre_team_cols + pre_opp_cols] = (
    base_eligible[pre_team_cols + pre_opp_cols].fillna(0).astype(int)
)

# --- pre-window team-vs-opponent diffs --------------------------------------
# Symmetric with the half-time `ht_diff_n_*` features so W and X share the
# same schema, differing only in the time window.
for m in metric_cols:
    base_eligible[f"pre_diff_{m}"] = (
        base_eligible[f"pre_team_{m}"] - base_eligible[f"pre_opp_{m}"]
    )
pre_diff_cols = [f"pre_diff_{m}" for m in metric_cols]

# --- pre-window score differential ------------------------------------------
pre_goals = build_window_goals(events, "pre")
own_g = pre_goals.rename(columns={"goals": "_pre_goals_for"})
opp_g = pre_goals.rename(columns={"team_id": "opponent_team_id",
                                  "goals":   "_pre_goals_against"})
base_eligible = (
    base_eligible
    .merge(own_g, on=["match_id", "team_id"],          how="left")
    .merge(opp_g, on=["match_id", "opponent_team_id"], how="left")
)
base_eligible[["_pre_goals_for", "_pre_goals_against"]] = (
    base_eligible[["_pre_goals_for", "_pre_goals_against"]].fillna(0).astype(int)
)
base_eligible["pre_score_diff"] = (
    base_eligible["_pre_goals_for"] - base_eligible["_pre_goals_against"]
)
base_eligible = base_eligible.drop(
    columns=["_pre_goals_for", "_pre_goals_against", "opponent_team_id"]
)

print(f"Excluded event types: {sorted(PRE_EXCLUDE_EVENT_TYPES)}")
print(f"Added {len(pre_team_cols)} team + {len(pre_opp_cols)} opponent pre-window columns "
      f"+ {len(pre_diff_cols)} team-vs-opp diff columns + 1 score diff column")
print(f"\n{'event type':<22} {'team mean':>10} {'opp mean':>10} {'diff mean':>10}")
print("-" * 56)
for tcol, ocol, dcol in zip(pre_team_cols, pre_opp_cols, pre_diff_cols):
    label = tcol.replace("pre_team_n_", "") or "events"
    print(f"{label:<22} {base_eligible[tcol].mean():>10.2f} "
          f"{base_eligible[ocol].mean():>10.2f} {base_eligible[dcol].mean():>10.3f}")
print(f"\npre_score_diff: mean={base_eligible['pre_score_diff'].mean():.3f}, "
      f"std={base_eligible['pre_score_diff'].std():.3f}, "
      f"min={base_eligible['pre_score_diff'].min()}, max={base_eligible['pre_score_diff'].max()}")


Excluded event types: ['Camera On', 'Camera off', 'Half Start', 'Starting XI']
Added 31 team + 31 opponent pre-window columns + 31 team-vs-opp diff columns + 1 score diff column

event type              team mean   opp mean  diff mean
--------------------------------------------------------
50_50                        0.27       0.27     -0.000
aerial_lost                  2.90       2.90     -0.005
bad_behaviour                0.01       0.01     -0.001
ball_receipt                82.05      81.62      0.426
ball_recovery                8.51       8.50      0.012
block                        3.17       3.18     -0.012
carry                       67.09      66.70      0.391
clearance                    3.44       3.46     -0.023
dispossessed                 2.15       2.14      0.004
dribble                      2.71       2.70      0.015
dribbled_past                1.65       1.66     -0.010
error                        0.04       0.04     -0.001
foul_committed               2.09   

### 5.2 Pre-window event intensity (player)

Same per-type counts but for the focal player only (`pre_player_*`). Proxy for individual involvement in the pre window.

In [8]:
def build_pre_player_intensity(events: pd.DataFrame) -> pd.DataFrame:
    """(match_id, player_id) event-type counts in the pre window.

    Mirrors `build_pre_team_intensity` but keyed on the individual player.
    Returns one row per (match_id, player_id) with `pre_player_n_events` and
    one `pre_player_n_<type>` column per non-excluded event type.
    """
    early = events.loc[
        in_window(events, "pre")
        & ~events["type"].isin(PRE_EXCLUDE_EVENT_TYPES)
    ].dropna(subset=["player_id"])

    by_type = (
        early.groupby(["match_id", "player_id", "type"]).size()
             .unstack(fill_value=0)
    )
    by_type.columns = [f"pre_player_n_{clean_event_name(c)}" for c in by_type.columns]
    return by_type.reset_index()


pre_player = build_pre_player_intensity(events)
pre_player_cols = [c for c in pre_player.columns if c.startswith("pre_player_")]

base_eligible = base_eligible.merge(pre_player, on=["match_id", "player_id"], how="left")
# Players with zero pre-window events get 0, not NaN
base_eligible[pre_player_cols] = base_eligible[pre_player_cols].fillna(0).astype(int)

print(f"Added {len(pre_player_cols)} player-level pre-window columns")
print(f"\n{'event type':<22} {'player mean':>12} {'player max':>12} {'share zero':>11}")
print("-" * 60)
for c in pre_player_cols:
    label = c.replace("pre_player_n_", "") or "events"
    mean = base_eligible[c].mean()
    mx   = base_eligible[c].max()
    zero = (base_eligible[c] == 0).mean()
    print(f"{label:<22} {mean:>12.2f} {mx:>12} {zero:>10.1%}")

Added 29 player-level pre-window columns

event type              player mean   player max  share zero
------------------------------------------------------------
50_50                          0.02            3      98.3%
aerial_lost                    0.20            7      84.4%
bad_behaviour                  0.00            0     100.0%
ball_receipt                   5.61           46      26.2%
ball_recovery                  0.59            7      60.1%
block                          0.21            5      82.2%
carry                          4.61           45      26.4%
clearance                      0.24            8      82.6%
dispossessed                   0.14            5      87.9%
dribble                        0.18            8      85.9%
dribbled_past                  0.11            4      90.2%
error                          0.00            2      99.7%
foul_committed                 0.13            4      88.1%
foul_won                       0.13            5      88

## 6. Team-level covariates

In [9]:
def build_team_covariates(matches: pd.DataFrame) -> pd.DataFrame:
    """(match_id, team_id)-keyed: team_name, opponent_name."""
    home = matches[["match_id", "home_team_id", "home_team", "away_team"]].rename(
        columns={"home_team_id": "team_id", "home_team": "team_name", "away_team": "opponent_name"}
    )
    away = matches[["match_id", "away_team_id", "away_team", "home_team"]].rename(
        columns={"away_team_id": "team_id", "away_team": "team_name", "home_team": "opponent_name"}
    )
    return pd.concat([home, away], ignore_index=True)


team_covs = build_team_covariates(matches)
base_eligible = base_eligible.merge(team_covs, on=["match_id", "team_id"], how="left")

print(f"team_name nulls: {base_eligible['team_name'].isna().sum()}")
print("\nTop teams:")
print(base_eligible["team_name"].value_counts().head(10).to_string())

team_name nulls: 0

Top teams:
team_name
Barcelona              6698
Paris Saint-Germain    1246
Manchester United      1001
Arsenal                 968
Bayer Leverkusen        879
Real Madrid             872
Atlético Madrid         830
Sevilla                 807
Athletic Club           800
Valencia                791


## 7. Half-time game state

Player-team context at the end of period 1 (minute 45). Used downstream as
heterogeneity / projection features for the CATE driver analysis.

Caveat: minute 45 sits at the **end** of the treatment window, so these are
strictly post-treatment variables. They're appropriate for projecting the
CATE onto interpretable context (BLP-style diagnostic), but should be used
with care if folded into the CATE model itself.

Columns added:

* `home_away` — "home" / "away" (from `matches`)
* `ht_score_diff` — own goals − opp goals scored in period 1
* `ht_diff_n_events`, `ht_diff_n_off_events`, `ht_diff_n_def_events` — own − opp
  period-1 event counts (total, offensive, defensive)

The offensive set is the wide in-possession set minus Miscontrol.


In [10]:
OFFENSIVE_EVENT_TYPES = [
    "Pass",
    "Carry",
    "Ball Receipt*",
    "Dribble",
    "Shot",
    "Foul Won",
]


def build_home_away(matches: pd.DataFrame) -> pd.DataFrame:
    """(match_id, team_id) -> 'home' / 'away'."""
    home = matches[["match_id", "home_team_id"]].rename(
        columns={"home_team_id": "team_id"}).assign(home_away="home")
    away = matches[["match_id", "away_team_id"]].rename(
        columns={"away_team_id": "team_id"}).assign(home_away="away")
    return pd.concat([home, away], ignore_index=True)


def build_first_half_goals(events: pd.DataFrame) -> pd.DataFrame:
    """(match_id, team_id) -> goals scored in period 1.

    Counts shot goals (Shot with shot_outcome=='Goal') and 'Own Goal For'
    events (StatsBomb codes the benefiting team in `team_id`)."""
    p1 = events.loc[events["period"] == 1]
    shot_goals = p1.loc[
        (p1["type"] == "Shot") & (p1["shot_outcome"] == "Goal"),
        ["match_id", "team_id"],
    ]
    og_for = p1.loc[p1["type"] == "Own Goal For", ["match_id", "team_id"]]
    return (
        pd.concat([shot_goals, og_for], ignore_index=True)
          .groupby(["match_id", "team_id"]).size()
          .reset_index(name="goals_ht")
    )


def build_first_half_intensity(events: pd.DataFrame) -> pd.DataFrame:
    """(match_id, team_id) -> period-1 event counts.

    Returns one row per (match_id, team_id) with:
      * `n_events`, `n_off_events`, `n_def_events` — aggregates,
      * one `n_<type>` column per non-excluded event type — used
        downstream to build per-type team-vs-opponent diffs as
        half-time-state heterogeneity features.

    Vectorized via a single (match_id, team_id, type) crosstab — avoids
    the per-group Python lambdas that scale catastrophically on a
    ~7,000-group × ~1,700-event-per-group panel.
    """
    p1 = events.loc[
        (events["period"] == 1)
        & ~events["type"].isin(PRE_EXCLUDE_EVENT_TYPES)
    ]
    wide = (
        p1.groupby(["match_id", "team_id", "type"], sort=False).size()
          .unstack(fill_value=0)
    )
    wide.columns = [f"n_{clean_event_name(c)}" for c in wide.columns]
    return wide.reset_index()


def build_first_half_player_intensity(events: pd.DataFrame) -> pd.DataFrame:
    """(match_id, player_id) -> period-1 player-level event counts.

    Mirrors `build_pre_player_intensity` but on the full first half (period 1)
    instead of [0, 15). Returns one `ht_player_n_<type>` column per non-
    excluded event type, plus an `ht_player_n_events` total and the two
    `ht_player_n_def_events` / `ht_player_n_off_events` aggregates (same
    DEFENSIVE_EVENT_TYPES / OFFENSIVE_EVENT_TYPES used everywhere else).
    All serve as half-time-state moderators in the CATE second stage.
    """
    p1 = events.loc[
        (events["period"] == 1)
        & ~events["type"].isin(PRE_EXCLUDE_EVENT_TYPES)
    ].dropna(subset=["player_id"])

    wide = (
        p1.groupby(["match_id", "player_id", "type"], sort=False).size()
          .unstack(fill_value=0)
    )
    wide.columns = [f"ht_player_n_{clean_event_name(c)}" for c in wide.columns]
    # Drop the event types a yellow card attaches to: the carded foul /
    # bad-behaviour in [15, 45] mechanically encodes the treatment, so these
    # player-level half-time counts would leak T into the CATE moderators.
    leaky = ["ht_player_n_foul_committed", "ht_player_n_bad_behaviour"]
    wide = wide.drop(columns=[c for c in leaky if c in wide.columns])
    return wide.reset_index()


# --- home/away -----------------------------------------------------------
ha = build_home_away(matches)
base_eligible = base_eligible.merge(ha, on=["match_id", "team_id"], how="left")

# --- score_diff at HT ----------------------------------------------------
opp_map = _opponent_team_map(matches)
goals   = build_first_half_goals(events)

own_goals = goals.rename(columns={"goals_ht": "_ht_goals_for"})
opp_goals = goals.rename(columns={
    "team_id": "opponent_team_id", "goals_ht": "_ht_goals_against",
})

base_eligible = (
    base_eligible
    .merge(opp_map,   on=["match_id", "team_id"],          how="left")
    .merge(own_goals, on=["match_id", "team_id"],          how="left")
    .merge(opp_goals, on=["match_id", "opponent_team_id"], how="left")
)
base_eligible[["_ht_goals_for", "_ht_goals_against"]] = (
    base_eligible[["_ht_goals_for", "_ht_goals_against"]].fillna(0).astype(int)
)
base_eligible["ht_score_diff"] = (
    base_eligible["_ht_goals_for"] - base_eligible["_ht_goals_against"]
)

# --- event-count differentials at HT (team) ------------------------------
fh = build_first_half_intensity(events)
metric_cols_ht = [c for c in fh.columns if c.startswith("n_")]

own_fh = fh.rename(columns={c: f"_ht_team_{c}" for c in metric_cols_ht})
opp_fh = fh.rename(columns={"team_id": "opponent_team_id",
                            **{c: f"_ht_opp_{c}" for c in metric_cols_ht}})
base_eligible = (
    base_eligible
    .merge(own_fh, on=["match_id", "team_id"],          how="left")
    .merge(opp_fh, on=["match_id", "opponent_team_id"], how="left")
)

for c in metric_cols_ht:
    own = base_eligible[f"_ht_team_{c}"].fillna(0).astype(int)
    opp = base_eligible[f"_ht_opp_{c}"].fillna(0).astype(int)
    base_eligible[f"ht_diff_{c}"] = own - opp

# --- player-level halftime intensity (full per-type set) -----------------
fh_player = build_first_half_player_intensity(events)
ht_player_cols = [c for c in fh_player.columns if c.startswith("ht_player_")]
base_eligible = base_eligible.merge(
    fh_player, on=["match_id", "player_id"], how="left"
)
base_eligible[ht_player_cols] = (
    base_eligible[ht_player_cols].fillna(0).astype(int)
)

# Drop intermediates
intermediates = [c for c in base_eligible.columns if c.startswith("_ht_")] + ["opponent_team_id"]
base_eligible = base_eligible.drop(columns=intermediates)

# --- diagnostics ---------------------------------------------------------
print("home_away:")
print(base_eligible["home_away"].value_counts(dropna=False).to_string())

ht_diff_cols   = sorted(c for c in base_eligible.columns if c.startswith("ht_diff_n_"))
ht_player_cols = sorted(c for c in base_eligible.columns if c.startswith("ht_player_n_"))
print(f"\nht_diff_n_*   columns ({len(ht_diff_cols)})")
print(f"ht_player_n_* columns ({len(ht_player_cols)})")

print(f"\n{'metric':<30} {'mean':>9} {'std':>9} {'min':>6} {'max':>6}")
print("-" * 62)
for c in ["ht_score_diff"] + ht_diff_cols + ht_player_cols:
    s = base_eligible[c]
    print(f"{c:<30} {s.mean():>9.2f} {s.std():>9.2f} {s.min():>6} {s.max():>6}")


home_away:
home_away
home    43769
away    43388

ht_diff_n_*   columns (32)
ht_player_n_* columns (27)

metric                              mean       std    min    max
--------------------------------------------------------------
ht_score_diff                       0.02      1.25     -6      6
ht_diff_n_50_50                    -0.00      0.13     -2      2
ht_diff_n_aerial_lost              -0.02      4.72    -21     21
ht_diff_n_bad_behaviour            -0.00      0.38     -3      3
ht_diff_n_ball_receipt              1.20    140.98   -505    505
ht_diff_n_ball_recovery             0.04      7.59    -34     34
ht_diff_n_block                    -0.03      5.15    -20     20
ht_diff_n_carry                     1.15    130.37   -433    433
ht_diff_n_clearance                -0.05      8.08    -36     36
ht_diff_n_dispossessed              0.00      3.96    -16     16
ht_diff_n_dribble                   0.04      5.74    -27     27
ht_diff_n_dribbled_past            -0.02      4.29  

## 8. Player-level covariates

In [12]:
POSITION_GROUP = {
    "Goalkeeper"               : "Goalkeeper",
    "Left Back"                : "Defender",
    "Right Back"               : "Defender",
    "Center Back"              : "Defender",
    "Left Center Back"         : "Defender",
    "Right Center Back"        : "Defender",
    "Left Wing Back"           : "Defender",
    "Right Wing Back"          : "Defender",
    "Left Defensive Midfield"  : "Midfielder",
    "Right Defensive Midfield" : "Midfielder",
    "Center Defensive Midfield": "Midfielder",
    "Left Center Midfield"     : "Midfielder",
    "Right Center Midfield"    : "Midfielder",
    "Center Midfield"          : "Midfielder",
    "Left Midfield"            : "Midfielder",
    "Right Midfield"           : "Midfielder",
    "Left Attacking Midfield"  : "Midfielder",
    "Right Attacking Midfield" : "Midfielder",
    "Center Attacking Midfield": "Midfielder",
    "Left Wing"                : "Forward",
    "Right Wing"               : "Forward",
    "Left Center Forward"      : "Forward",
    "Right Center Forward"     : "Forward",
    "Center Forward"           : "Forward",
    "Secondary Striker"        : "Forward",
}

def build_player_covariates(events: pd.DataFrame) -> pd.DataFrame:
    """player_id-keyed: position, position_group."""
    pos = (
        events.dropna(subset=["player_id", "position"])
        .groupby("player_id")["position"]
        .agg(lambda s: s.mode().iloc[0])
        .reset_index()
    )
    pos["position_group"] = pos["position"].map(POSITION_GROUP)
    return pos


player_covs = build_player_covariates(events)
base_eligible = base_eligible.merge(player_covs, on="player_id", how="left")

print("Position group distribution:")
print(base_eligible["position_group"].value_counts().to_string())

Position group distribution:
position_group
Defender      29955
Midfielder    27421
Forward       22850
Goalkeeper     6925


## 9. Sample restriction

Restrict to outfield positions (`Defender`, `Midfielder`, `Forward`).
Goalkeepers are dropped because they produce almost no defensive events
in our sense. No further restriction on gender or competition is applied;
all such variables enter the analysis as controls.


In [13]:
before = len(base_eligible)

base_eligible = base_eligible[
    base_eligible["position_group"].isin(["Defender", "Midfielder", "Forward"])
].reset_index(drop=True)

print(f"Before restriction: {before:>7,} player-matches")
print(f"After  restriction: {len(base_eligible):>7,} player-matches")
print(f"Treatment rate    : {base_eligible['treat_yellow_card'].mean():.2%}")
print("\nBy gender / competition:")
print(base_eligible.groupby(["gender","competition"]).size().to_string())

# Drop zero-variance covariate columns (pre_*, ht_diff_n_*, ht_player_n_*).
covariate_prefixes = ("pre_", "ht_diff_n_", "ht_player_n_")
covariate_cols = [c for c in base_eligible.columns if c.startswith(covariate_prefixes)]
zero_var = [c for c in covariate_cols if base_eligible[c].nunique(dropna=False) <= 1]
if zero_var:
    base_eligible = base_eligible.drop(columns=zero_var)
print(f"\nDropped {len(zero_var)} zero-variance covariate columns:")
for c in zero_var:
    print(f"  - {c}")


Before restriction:  87,157 player-matches
After  restriction:  80,226 player-matches
Treatment rate    : 4.00%

By gender / competition:
gender  competition            
female  FA Women's Super League     7562
        NWSL                         848
        UEFA Women's Euro           1556
        Women's World Cup           2730
male    1. Bundesliga               7820
        African Cup of Nations      1290
        Champions League             402
        Copa America                 819
        Copa del Rey                  57
        FIFA U20 World Cup            22
        FIFA World Cup              3519
        Indian Super league         2841
        La Liga                    19807
        Liga Profesional              33
        Ligue 1                     9936
        Major League Soccer          148
        North American League         22
        Premier League              9677
        Serie A                     8429
        UEFA Euro                   2646
        UE

## 10. Final frame

In [14]:
KEY_COLS    = ["match_id", "player_id", "team_id"]
ENTITY_COLS = ["player_name", "team_name", "opponent_name",
               "position", "position_group", "home_away"]
MATCH_COLS  = ["competition_type", "competition", "season", "gender",
               "match_date", "match_week"]

pre_cols   = sorted(c for c in base_eligible.columns if c.startswith("pre_"))
ht_cols    = sorted(c for c in base_eligible.columns if c.startswith("ht_"))
treat_cols = sorted(c for c in base_eligible.columns if c.startswith("treat_"))
post_cols  = sorted(c for c in base_eligible.columns if c.startswith("post_"))

ordered = (KEY_COLS + ENTITY_COLS + MATCH_COLS
           + pre_cols + ht_cols + treat_cols + post_cols)
analysis = base_eligible[ordered].copy()
print(f"Analysis frame: {len(analysis):,} rows × {analysis.shape[1]} columns  "
      f"pre_*({len(pre_cols)})  ht_*({len(ht_cols)})  "
      f"treat_*({len(treat_cols)})  post_*({len(post_cols)})")
analysis.head()

Analysis frame: 80,226 rows × 227 columns
  pre_*(118)  ht_*(56)  str_*(17)  treat_*(1)  post_*(20)


,match_id,player_id,team_id,player_name,team_name,opponent_name,position,position_group,home_away,competition_type,...,post_n_error,post_n_events,post_n_foul_committed,post_n_foul_won,post_n_interception,post_n_off_events,post_n_pass,post_n_pressure,post_n_shot,post_n_tackle
0,3895302,34870.0,176,Nick Woltemade,Werder Bremen,Bayer Leverkusen,Right Center Forward,Forward,away,Domestic League,...,0,19,0,0,0,12,3,6,0,0
1,3895302,12299.0,176,Marvin Ducksch,Werder Bremen,Bayer Leverkusen,Left Center Forward,Forward,away,Domestic League,...,0,18,0,0,0,14,5,3,0,0
2,3895302,31100.0,176,Christian Groß,Werder Bremen,Bayer Leverkusen,Center Back,Defender,away,Domestic League,...,0,29,0,0,0,25,9,3,0,0
3,3895302,51769.0,176,Julián Malatini,Werder Bremen,Bayer Leverkusen,Right Center Back,Defender,away,Domestic League,...,0,35,0,0,1,28,11,1,0,1
4,3895302,8576.0,176,Mitchell Weiser,Werder Bremen,Bayer Leverkusen,Right Back,Defender,away,Domestic League,...,0,20,0,0,0,17,6,3,0,0


In [15]:
analysis.to_csv(DATA_DIR / "analysis_frame.csv", index=False)
print(f"Saved {len(analysis):,} rows × {analysis.shape[1]} columns  →  data/analysis_frame.csv")

Saved 80,226 rows × 227 columns  →  data/analysis_frame.csv
